# Code to deal with Prospector output files! Requires own environment

In [ ]:
import os
import glob
import numpy as np
import pickle as pkl
import pandas as pd
import prospect.io.read_results as reader
from prospect.utils.plotting import get_percentiles, get_best
from corner import quantile
import h5py


from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import WMAP9 as cosmo
from prospect.models.transforms import logsfr_ratios_to_sfrs
from prospect.sources import FastStepBasis
from astropy.cosmology import Planck18 as cosmo
from prospect.models.sedmodel import PolySpecModel, SpecModel


# Prospector Loading Function

In [ ]:
def load_prospector_results(galaxy_id, prosp_dir):
    """Function to load Prospector result .h5 files and disect its 
    data structure to be used for easy plotting.

    Args:
        galaxy_id (int): The ID of the galaxy for which to load results
        prosp_dir (str): The directory containing the Prospector output files

    Returns:
        dict: A comprehensive dictionary storing the parameters samples, MAP values and quantiles
    """
    
    # Load the h5 file for the given galaxy ID
    h5_files = glob.glob(os.path.join(prosp_dir, f'*{galaxy_id}*.h5'))
    
    try:
        h5_file = h5_files[0]
        print(f"Loading Prospector output file: {h5_file}")
    except IndexError:
        print(f"No PROSPECTOR results found for objid {galaxy_id}.")
        return None

    # Load PROSPECTOR results
    results, obs, model = reader.results_from(h5_file)
        
    # Now we have to exclude the last 3 parameters from the fit
    map_parameters = get_best(results)
    
    # Extract labels for parameters that were "free" (fitted)
    labels = map_parameters[0]

    # Build the MAP dictionary
    MAP = {}
    for a,b in zip(map_parameters[0], map_parameters[1]):
        MAP[a] = b
    
    # Extract chains, weights and the MAP index
    chain = results['chain']
    weights = results['weights']
    imax = np.argmax(results['lnprobability'])
    
    data = {
            'meta': {'labels': map_parameters[0], 'map_idx': imax, 'weights': weights},
            'params': {}
        }

    perc = get_percentiles(results, [16, 50, 84])    
    
    for i, name in enumerate(map_parameters[0]):
            # Use the flattened chain for statistics
            param_samples = chain[:, i]
            
            if name == 'dust2':         # convert optical depth to mag
                data['params'][name] = {
                'samples': param_samples * 1.086,
                'map': MAP[name] * 1.086,   # Use the value from the best vector directly
                'q16': perc[name][0] * 1.086,
                'q50': perc[name][1] * 1.086,
                'q84': perc[name][2] * 1.086
            }
                
            else:
                data['params'][name] = {
                    'samples': param_samples,
                    'map': MAP[name],   # Use the value from the best vector directly
                    'q16': perc[name][0],
                    'q50': perc[name][1],
                    'q84': perc[name][2]
                }
    return data

phot_table = './Phot_Table_MIRI.fits'
with fits.open(phot_table) as hdul:
    galaxy_ids = hdul[1].data['ID']

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'

data = load_prospector_results(12717, prosp_dir)

# Bagpipes Loading Function

In [ ]:
bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/no_fesc_with_miri/'

example_id = 7102

def load_bagpipes_results(galaxy_id, bagp_dir):
    """Function to load Bagpipes result .h5 files and disect its 
    data structure to be used for easy plotting.

    Args:
        galaxy_id (int): The ID of the galaxy for which to load results
        bagp_dir (str): The directory containing the Bagpipes output files

    Returns:
        dict: A comprehensive dictionary storing the parameters samples, MAP values and quantiles
    """
    
    file = os.path.join(bagp_dir, f'{galaxy_id}.h5')    # Only one file per galaxy
    
    print(f"Loading Bagpipes output file: {file}")
    
    with h5py.File(file, 'r') as results:
        
        # Get redshift of the source
        fit_str = results.attrs['fit_instructions']
        fit = eval(fit_str, {"np": np, "array": np.array})
        zred = fit['redshift']
        
        # Extract the sampling results
        chain = results['samples2d']
        
        # Get maximum likelihood inde
        imax = np.argmax(results['lnlike'])
        
        # Manually extracted the labels from the results
        labels = ['dsfr1', 'dsfr2', 'dsfr3', 'dsfr4', 'dsfr5', 'dsfr6', 'logmass', 'logzsol', 'dust2', 
                    'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
        
        # Build the data structure
        data = {
            'meta': {'labels': labels, 'map_idx': imax, 'weights': None},
            'params': {}
        }
        
        # Loop through each parameter
        for i, name in enumerate(labels):
            samples = chain[:, i]
            q16, q50, q84 = quantile(samples, [0.16, 0.5, 0.84], weights=None)
            
            if name == 'logzsol':   # Convert metallicity to log
                data['params'][name] = {
                    'samples': np.log10(samples),
                    'map': np.log10(samples[imax]),
                    'q16': np.log10(q16),
                    'q50': np.log10(q50),
                    'q84': np.log10(q84)
                }
            
            else:    
                data['params'][name] = {
                    'samples': samples,
                    'map': samples[imax],
                    'q16': q16,
                    'q50': q50,
                    'q84': q84
                }

        data['params']['zred'] = {'samples': None, 'map': zred, 'q16': zred, 'q50': zred, 'q84': zred}
        
    return data
    
example_res = load_bagpipes_results(21424, bagp_dir)
print(example_res)


# Load results and store them in pickle files

In [ ]:
# Create a directory for your "Analysis Ready" data
output_dir = './comparison/fesc_with_miri/pickles'
os.makedirs(output_dir, exist_ok=True)

miri_table = "./Phot_Table_MIRI.fits"
id_table = Table.read(miri_table)
all_ids = [int(val.decode('utf-8') if isinstance(val, bytes) else val) 
                        for val in id_table['ID']]

print(len(all_ids))

no_spec = [9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990]
bl_agn = [12020, 18977]
high_z = [11247, 7696]
intermediate_ap = [7136, 1904, 7922, 8469, 10314, 11337, 11420, 11451, 17517, 17669, 18332, 21452]

exclude = set(no_spec + bl_agn + intermediate_ap + high_z)

missing_from_table = [i for i in exclude if i not in set(all_ids)]

print(f"The following {len(missing_from_table)} IDs were in your 'exclude' list "
      f"but were NOT found in the MIRI table:\n{missing_from_table}")


print(len(exclude))
all_ids = [gal_id for gal_id in all_ids if gal_id not in exclude]

print(len(all_ids), "galaxies to process after exclusions.")

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'
bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/fesc_with_miri/'

for gal_id in all_ids:
    # 1. Load your existing functions
    try:
        p_data = load_prospector_results(gal_id, prosp_dir)
        b_data = load_bagpipes_results(gal_id, bagp_dir)
        
    except Exception as e:
        print(f"Error occurred while loading data for galaxy {gal_id}: {e}")
        continue

    # 2. Package them together
    combined_package = {
        'id': gal_id,
        'prospector': p_data,
        'bagpipes': b_data
    }

    # 3. Save to disk
    save_path = os.path.join(output_dir, f'{gal_id}_comp.pkl')
    with open(save_path, 'wb') as f:
        pkl.dump(combined_package, f)

# Plot the galaxies

In [ ]:
import corner
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np

import matplotlib as mpl
    
# Increase tick label size and thickness
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['axes.linewidth'] = 1.5  # Makes the box frames thicker

def plot_dual_corner(galaxy_id, comparison_dir):
    
    file = os.path.join(comparison_dir, f'pickles/{galaxy_id}_comp.pkl')  
    with open(file, 'rb') as f:
        data = pkl.load(f)
        
        # Load the individual fit results
        b_data = data['bagpipes']
        p_data = data['prospector']
    
    # Define internal keys and display labels
    # Make sure these match the keys in your b_data['params'] and p_data['params']
    plot_keys = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    labels = [r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", 
              r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", 
              r"Dust $U_{min}$", r"$\log_{10}(U)$"]

    # Extract and Transform Samples
    # Bagpipes
    b_samps = np.array([b_data['params'][k]['samples'] for k in plot_keys]).T
    # Prospector (Applying the Av conversion factor 1.086 to the 3rd column: index 2)
    p_samps = np.array([p_data['params'][k]['samples'] for k in plot_keys]).T
    
    samples_list = [b_samps, p_samps]
    sample_labels = ["Bagpipes", "Prospector"]
    colors = ["orange", "dodgerblue"]

    # Calculate Global Range (So both fits are visible)
    ndim = b_samps.shape[1]
    plot_range = []
    for dim in range(ndim):
        dim_min = min(np.nanmin(b_samps[:, dim]), np.nanmin(p_samps[:, dim]))
        dim_max = max(np.nanmax(b_samps[:, dim]), np.nanmax(p_samps[:, dim]))
        
        plot_range.append([dim_min, dim_max])

    # 4. Handle Weights 
    # Prospector weights from Dynesty
    p_weights = p_data['meta']['weights']
    # Bagpipes weights (Usually None/Equal, so create ones)
    b_weights = np.ones(len(b_samps))
    
    # Normalize weights so histograms have comparable heights
    # (Matching the logic from your provided demo script)
    b_weights *= (len(p_samps) / len(b_samps))

    # 5. Base Corner Settings
    shared_kwargs = dict(
        labels=labels,
        range=plot_range,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True,
        show_titles=False, # We disable automatic titles to avoid overlaps
        max_n_ticks=3,
        hist_kwargs=dict(density=True)
    )

    # 6. Plotting
    # First: Bagpipes
    fig = corner.corner(
        b_samps,
        labels=labels,
        range=plot_range,
        color="orange",
        label_kwargs={"fontsize": 14},
        weights=b_weights,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True,
        show_titles=False,
        max_n_ticks=4,
        hist_kwargs={'color': 'orange', 'linewidth': 2, 'density': True}
    )

    # Second: Prospector
    # We turn off 'fill_contours' for the second one so we can see through it
    corner.corner(
        p_samps,
        fig=fig,
        range=plot_range,
        color="dodgerblue",
        label_kwargs={"fontsize": 14}, 
        weights=p_weights,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True, 
        show_titles=False,
        max_n_ticks=4,
        hist_kwargs={'color': 'dodgerblue', 'linewidth': 2, 'density': True}
    )

    # 7. Add Legend and Title
    plt.legend(
        handles=[
            mlines.Line2D([], [], color=colors[i], label=sample_labels[i], lw=4)
            for i in range(len(colors))
        ],
        fontsize=20, frameon=False,
        bbox_to_anchor=(1, ndim), loc="upper right"
    )
    
    ndim = b_samps.shape[1]
    axes = np.array(fig.axes).reshape((ndim, ndim))
    
    for i in range(ndim):
        ax = axes[i, i]
        
        # 2. Extract stats for Bagpipes (Orange)
        b_p = b_data['params'][plot_keys[i]]
        b_val, b_plus, b_minus = b_p['q50'], b_p['q84'] - b_p['q50'], b_p['q50'] - b_p['q16']
        
        # 3. Extract stats for Prospector (Black/Blue)
        p_p = p_data['params'][plot_keys[i]]
        p_val, p_plus, p_minus = p_p['q50'], p_p['q84'] - p_p['q50'], p_p['q50'] - p_p['q16']
        
        # Special Case: If it's Av (index 2), apply the 1.086 scale to the text labels too
        if i == 2:
            b_val, b_plus, b_minus = b_val, b_plus, b_minus # Bagpipes is already Av
            p_val, p_plus, p_minus = p_val*1.086, p_plus*1.086, p_minus*1.086

        # 4. Create the strings
        # Use \text{} or raw strings to handle the LaTeX formatting
        b_str = f"${b_val:.2f}^{{+{b_plus:.2f}}}_{{-{b_minus:.2f}}}$"
        p_str = f"${p_val:.2f}^{{+{p_plus:.2f}}}_{{-{p_minus:.2f}}}$"

        # 5. Set the title
        # We use a newline \n to stack them. Note: 'y' controls the vertical height.
        # 4. Place individual text objects (Manually colored)
        # x=0.5 centers it. y=1.02 and 1.15 stack them above the plot.
        ax.text(0.5, 1.15, b_str, color="orange", transform=ax.transAxes, 
                fontsize=14, ha='center', va='bottom', fontweight='bold')
        
        ax.text(0.5, 1.02, p_str, color="dodgerblue", transform=ax.transAxes, 
                fontsize=14, ha='center', va='bottom', fontweight='bold')
        
        # 6. Coloring the text
        # To get specific colors for specific lines of the title, 
        # we can use ax.annotate or just rely on the labels in the legend.
        # But for absolute clarity, we can color the whole title block:
        #ax.title.set_color('black') # Or 'darkgrey' to be neutral
    
    fig.suptitle(f"Bagpipes vs. Prospector\nGalaxy {galaxy_id}", fontsize=24, y=1.0)
    
    fig_path = os.path.join(comparison_dir, 'plots', f'{galaxy_id}_corner.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close()

# Create corner plots for the whole sample!

In [ ]:
comp_dir = './comparison/fesc_with_miri/'

pickle_files = glob.glob(os.path.join(comp_dir, 'pickles', '*_comp.pkl'))

for p in pickle_files:
    objid = int(os.path.basename(p).split('_comp.pkl')[0])  # Extract the galaxy ID from the filename
    plot_dual_corner(objid, comp_dir)
    print(f"Plotted corner plot for galaxy ID: {objid}")

# Look at a single Bagpipes output

In [ ]:
def plot_bagpipes_corner(galaxy_id, b_data):
    # Select the labels you want to see
    # Using your existing labels: ['logmass', 'logzsol', 'dust2', 'gas_logu']
    plot_labels = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    # Extract the samples for these specific labels
    samples = np.array([b_data['params'][l]['samples'] for l in plot_labels]).T
    
    # Create the corner plot
    fig = corner.corner(
        samples,
        labels=[r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", r"Dust $U_{min}$", r"Dust $U_{min}$", r"$\log_{10}(U)$"],
        quantiles=[0.16, 0.5, 0.84],
        weights=b_data['meta']['weights'],
        show_titles=True,
        title_kwargs={"fontsize": 12},
        color="Orange",
        smooth=1.0, # Helps visualize bimodality with only 500 samples
        # --- ADD THESE THREE LINES ---
        plot_datapoints=False,  # Suppresses the black dots
        fill_contours=True,     # Fills the 1/2/3 sigma levels with color
        plot_density=False      # Suppresses the grey background 'cloud'
    )
    
    fig.suptitle(f"Galaxy {galaxy_id} Bagpipes", fontsize=16)
    plt.show()

example_res = load_bagpipes_results(21424, bagp_dir)

plot_bagpipes_corner(21424, example_res)

# Plot Prospector Corner Plot

In [ ]:
def plot_prospector_corner(galaxy_id, p_data):
    # Select the labels you want to see
    # Using your existing labels: ['logmass', 'logzsol', 'dust2', 'gas_logu']
    plot_labels = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    # Extract the samples for these specific labels
    samples = np.array([p_data['params'][l]['samples'] for l in plot_labels]).T
    
    # Create the corner plot
    fig = corner.corner(
        samples,
        labels=[r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", r"Dust $U_{min}$", r"Dust $U_{min}$", r"$\log_{10}(U)$"],
        quantiles=[0.16, 0.5, 0.84],
        weights=p_data['meta']['weights'],
        show_titles=True,
        title_kwargs={"fontsize": 12},
        color="black",
        smooth=1.0, # Helps visualize bimodality with only 500 samples
        # --- ADD THESE THREE LINES ---
        plot_datapoints=False,  # Suppresses the black dots
        fill_contours=True,     # Fills the 1/2/3 sigma levels with color
        plot_density=False      # Suppresses the grey background 'cloud'
    )
    fig.suptitle(f"Galaxy {galaxy_id} Prospector", fontsize=16)
    plt.show()

example_res = load_prospector_results(21424, prosp_dir)

plot_prospector_corner(21424, example_res)

# Example of a Corner plot that works

need to draw tallest histogram last since y-axis is constantly rescaled

In [ ]:
"""Demo to overlay multiple corners on top of each other"""
import corner
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np

CORNER_KWARGS = dict(
    smooth=0.9,
    label_kwargs=dict(fontsize=16),
    title_kwargs=dict(fontsize=16),
    quantiles=[0.16, 0.5, 0.84],
    levels=(1 - np.exp(-0.5), 1 - np.exp(-2), 1 - np.exp(-9 / 2.)),
    plot_density=False,
    plot_datapoints=False,
    fill_contours=True,
    show_titles=True,
    title_quantiles=[0.16, 0.5, 0.84],
    max_n_ticks=3,
)


def overlaid_corner(samples_list, sample_labels):
    """Plots multiple corners on top of each other"""
    # get some constants
    n = len(samples_list)
    _, ndim = samples_list[0].shape
    max_len = max([len(s) for s in samples_list])
    cmap = plt.cm.get_cmap('gist_rainbow', n)
    colors = [cmap(i) for i in range(n)]

    plot_range = []
    for dim in range(ndim):
        plot_range.append(
            [
                min([min(samples_list[i].T[dim]) for i in range(n)]),
                max([max(samples_list[i].T[dim]) for i in range(n)]),
            ]
        )

    CORNER_KWARGS.update(range=plot_range)

    fig = corner.corner(
        samples_list[0],
        color=colors[0],
        **CORNER_KWARGS
    )

    for idx in range(1, n):
        fig = corner.corner(
            samples_list[idx],
            fig=fig,
            weights=get_normalisation_weight(len(samples_list[idx]), max_len),
            color=colors[idx],
            **CORNER_KWARGS
        )

    plt.legend(
        handles=[
            mlines.Line2D([], [], color=colors[i], label=sample_labels[i])
            for i in range(n)
        ],
        fontsize=20, frameon=False,
        bbox_to_anchor=(1, ndim), loc="upper right"
    )
    plt.savefig("corner.png")
    plt.close()


def get_normalisation_weight(len_current_samples, len_of_longest_samples):
    return np.ones(len_current_samples) * (len_of_longest_samples / len_current_samples)


def main():
    ndim, nsamples = 3, 10000
    samples = np.random.randn(ndim * nsamples).reshape([nsamples, ndim])

    overlaid_corner(
        [samples*3, samples*2, samples],
        ["samples x 3", "samples x 2", "samples"]
    )


if __name__ == "__main__":
    main()

In [ ]:
import corner
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np

def plot_dual_corner(galaxy_id, b_data, p_data):
    # 1. Define internal keys and display labels
    # Make sure these match the keys in your b_data['params'] and p_data['params']
    plot_keys = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    labels = [r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", 
              r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", 
              r"Dust $U_{min}$", r"$\log_{10}(U)$"]

    # 2. Extract and Transform Samples
    # Bagpipes
    b_samps = np.array([b_data['params'][k]['samples'] for k in plot_keys]).T
    # Prospector (Applying the Av conversion factor 1.086 to the 3rd column: index 2)
    p_samps = np.array([p_data['params'][k]['samples'] for k in plot_keys]).T
    
    samples_list = [b_samps, p_samps]
    sample_labels = ["Bagpipes", "Prospector"]
    colors = ["orange", "dodgerblue"]

    # 3. Calculate Global Range (So both fits are visible)
    ndim = b_samps.shape[1]
    plot_range = []
    for dim in range(ndim):
        dim_min = min(np.nanmin(b_samps[:, dim]), np.nanmin(p_samps[:, dim]))
        dim_max = max(np.nanmax(b_samps[:, dim]), np.nanmax(p_samps[:, dim]))
        # Add 5% padding
        span = dim_max - dim_min
        plot_range.append([dim_min - 0.05*span, dim_max + 0.05*span])

    print(plot_range)

    plot_range = []
    for dim in range(ndim):
        # Combine samples to find a shared 95% window
        combined = np.concatenate([b_samps[:, dim], p_samps[:, dim]])
        # Remove NaNs and Infs for percentile calculation
        combined = combined[np.isfinite(combined)]
        
        vmin = np.percentile(combined, 0)
        vmax = np.percentile(combined, 100)
        plot_range.append([vmin, vmax])

    print(plot_range)

    # 4. Handle Weights 
    # Prospector weights from Dynesty
    p_weights = p_data['meta']['weights']
    # Bagpipes weights (Usually None/Equal, so create ones)
    b_weights = np.ones(len(b_samps))
    
    # Normalize weights so histograms have comparable heights
    # (Matching the logic from your provided demo script)
    b_weights *= (len(p_samps) / len(b_samps))

    # 5. Base Corner Settings
    shared_kwargs = dict(
        labels=labels,
        range=plot_range,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True,
        show_titles=False, # We disable automatic titles to avoid overlaps
        max_n_ticks=3,
        hist_kwargs=dict(density=True)
    )

    # 6. Plotting
    # First: Bagpipes
    fig = corner.corner(
        b_samps,
        labels=labels,
        range=plot_range,
        color="orange",
        weights=b_weights,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True,
        show_titles=True,
        max_n_ticks=4,
        hist_kwargs={'color': 'orange', 'linewidth': 2, 'density': True}
    )

    # Second: Prospector
    # We turn off 'fill_contours' for the second one so we can see through it
    corner.corner(
        p_samps,
        fig=fig,
        range=plot_range,
        color="dodgerblue",
        weights=p_weights,
        smooth=0.9,
        quantiles=[0.16, 0.5, 0.84],
        plot_density=False,
        plot_datapoints=False,
        fill_contours=True, 
        show_titles=False,
        max_n_ticks=4,
        hist_kwargs={'color': 'dodgerblue', 'linewidth': 2, 'linestyle': '-', 'density': True, 'alpha': 0.7}
    )

    # 7. Add Legend and Title
    plt.legend(
        handles=[
            mlines.Line2D([], [], color=colors[i], label=sample_labels[i], lw=2)
            for i in range(len(colors))
        ],
        fontsize=16, frameon=False,
        bbox_to_anchor=(1, ndim), loc="upper right"
    )
    
    fig.suptitle(f"Galaxy {galaxy_id} Comparison", fontsize=22, y=1.02)
    
    return fig

b_data = load_bagpipes_results(21424, bagp_dir)
p_data = load_prospector_results(21424, prosp_dir)

fig = plot_dual_corner(21424, b_data, p_data)